# generatemappingtext

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
MTR Peptidetextdatatext

target:
1. extractallsamplepeptide, MHC, microbesinformation
2. generatededuplicatepeptide+MHCtext
3. savecomplete mappingtext, textresultstext

Output files:
- unique_peptide_mhc.csv: deduplicatepeptide+MHCtext(text)
- peptide_full_mapping.csv: textmappingtext
- peptide_mapping_summary.json: statistics
"""

import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
import json
from collections import defaultdict
import pickle
from datetime import datetime

# ============================================================================
# text
# ============================================================================

# cancer-type list
CANCER_TYPES = ["BRCA", "CRC", "LUNG", "OSCC", "BLCA", "STAD", "LIHC", "PAAD", "CESC"]

# base path
BASE_DIR = "/path/to/project/results_V3/cancers_V3.1"

# output directory
OUTPUT_DIR = "/path/to/project/results_V3/summary/MTR_peptide_imm"

# filetext
FILE_PATTERN = "06.antigen/06.2.MTR_peptides/Tumor/*_classI_filtered.tsv"

# ============================================================================
# text
# ============================================================================

def get_sample_files(cancer_type):
    """textcancer typetextallsamplefile"""
    pattern = os.path.join(BASE_DIR, cancer_type, FILE_PATTERN)
    files = glob.glob(pattern)
    return files


def extract_sample_name(file_path):
    """fromfilepathextractsampletext"""
    basename = os.path.basename(file_path)
    sample_name = basename.replace("_classI_filtered.tsv", "")
    return sample_name


def load_sample_data(file_path, cancer_type):
    """
    loadtextsamplesdata
    Returns:
    DataFrame with columns: sample, cancer_type, peptide, MHC, microbes
    """
    try:
        df = pd.read_csv(file_path, sep='\t')
        # extractsampletext
        sample_name = extract_sample_name(file_path)
        # textcolumn
        required_cols = ['peptide', 'MHC', 'microbes']
        # checkcolumntextexists
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"Warning️ warning: {file_path} textcolumn: {missing_cols}")
            return None
        # extracttextcolumn
        subset = df[required_cols].copy()
        subset['sample'] = sample_name
        subset['cancer_type'] = cancer_type
        # textcolumncolumntext
        subset = subset[['sample', 'cancer_type', 'peptide', 'MHC', 'microbes']]
        return subset
    except Exception as e:
        print(f"Failed error: notextread {file_path}: {e}")
        return None


def create_unique_key(row):
    """createpeptide+MHCuniquetext"""
    return f"{row['peptide']}||{row['MHC']}"


def process_all_data():
    """processallcancer typeallsample counttext"""
    print("="*80)
    print("MTR Peptidetextdatatext")
    print("="*80)
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"cancer typetext: {len(CANCER_TYPES)}")
    print(f"output directory: {OUTPUT_DIR}")
    print("="*80)
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    # textalldata
    all_data = []
    # statistics
    stats = {
        'cancer_types': {},
        'total_samples': 0,
        'total_records': 0,
        'unique_peptide_mhc': 0,
        'unique_peptides': 0,
        'unique_mhc': 0
    }
    # processeachcancer type
    for cancer_type in CANCER_TYPES:
        print(f"\nprocess {cancer_type}...")
        # textcancer typeallsamplefile
        sample_files = get_sample_files(cancer_type)
        if not sample_files:
            print(f" Warning️ not foundsamplefile")
            continue
        print(f" found {len(sample_files)} samples")
        cancer_data = []
        for file_path in sample_files:
            data = load_sample_data(file_path, cancer_type)
            if data is not None and len(data) > 0:
                cancer_data.append(data)
                if cancer_data:
                    cancer_df = pd.concat(cancer_data, ignore_index=True)
                    all_data.append(cancer_df)
                    # statistics
                    stats['cancer_types'][cancer_type] = {
                        'samples': len(sample_files),
                        'records': len(cancer_df),
                        'unique_peptides': cancer_df['peptide'].nunique(),
                        'unique_mhc': cancer_df['MHC'].nunique()
                    }
                    print(f" OK load {len(sample_files)} samples, {len(cancer_df)} records")
                    # textalldata
                    if not all_data:
                        print("Failed error: textloadtextdata!")
                        return
                    print("\n" + "="*80)
                    print("textalldata...")
                    full_data = pd.concat(all_data, ignore_index=True)
                    print(f"OK total records: {len(full_data)}")
                    print(f"OK textsample count: {full_data['sample'].nunique()}")
                    print(f"OK total cancer types: {full_data['cancer_type'].nunique()}")
                    # Updatestatistics
                    stats['total_samples'] = full_data['sample'].nunique()
                    stats['total_records'] = len(full_data)
                    stats['unique_peptides'] = full_data['peptide'].nunique()
                    stats['unique_mhc'] = full_data['MHC'].nunique()
                    # ========================================================================
                    # 1. savecomplete mappingtext
                    # ========================================================================
                    print("\n" + "="*80)
                    print("savecomplete mappingtext...")
                    mapping_file = os.path.join(OUTPUT_DIR, "peptide_full_mapping.csv")
                    full_data.to_csv(mapping_file, index=False)
                    print(f"OK complete mappingtextsaved: {mapping_file}")
                    print(f" - recordtext: {len(full_data)}")
                    # textsaveforpickleformat(loadtext)
                    pickle_file = os.path.join(OUTPUT_DIR, "peptide_full_mapping.pkl")
                    full_data.to_pickle(pickle_file)
                    print(f"OK Pickleformatsaved: {pickle_file}")
                    # ========================================================================
                    # 2. generatededuplicatepeptide+MHCtext
                    # ========================================================================
                    print("\n" + "="*80)
                    print("generatededuplicatepeptide+MHCtext...")
                    # createuniquetext
                    full_data['peptide_mhc_key'] = full_data.apply(create_unique_key, axis=1)
                    # deduplicate
                    unique_peptide_mhc = full_data[['peptide', 'MHC']].drop_duplicates()
                    unique_peptide_mhc = unique_peptide_mhc.reset_index(drop=True)
                    # textID(textaftertextmapping)
                    unique_peptide_mhc.insert(0, 'peptide_mhc_id', range(1, len(unique_peptide_mhc) + 1))
                    print(f"OK deduplicateafterpeptide+MHCtext: {len(unique_peptide_mhc)}")
                    stats['unique_peptide_mhc'] = len(unique_peptide_mhc)
                    # savededuplicatelist(text)
                    unique_file = os.path.join(OUTPUT_DIR, "unique_peptide_mhc_for_prediction.csv")
                    unique_peptide_mhc.to_csv(unique_file, index=False)
                    print(f"OK deduplicatelistsaved: {unique_file}")
                    # ========================================================================
                    # 3. createtextmapping(peptide+MHC -> sampleinformation)
                    # ========================================================================
                    print("\n" + "="*80)
                    print("createtextmappingtext...")
                    # foreachpeptide+MHCtext, savealltextsampleandmicrobeinformation
                    reverse_mapping = defaultdict(list)
                    for _, row in full_data.iterrows():
                        key = create_unique_key(row)
                        reverse_mapping[key].append({
                            'sample': row['sample'],
                            'cancer_type': row['cancer_type'],
                            'microbes': row['microbes']
                        })
                        # savetextmapping
                        reverse_mapping_file = os.path.join(OUTPUT_DIR, "reverse_mapping.pkl")
                        with open(reverse_mapping_file, 'wb') as f:
                            pickle.dump(dict(reverse_mapping), f)
                            print(f"OK textmappingsaved: {reverse_mapping_file}")
                            print(f" - uniquepeptide+MHCtext: {len(reverse_mapping)}")
                            # ========================================================================
                            # 4. savestatistics
                            # ========================================================================
                            print("\n" + "="*80)
                            print("savestatistics...")
                            stats_file = os.path.join(OUTPUT_DIR, "peptide_mapping_summary.json")
                            with open(stats_file, 'w') as f:
                                json.dump(stats, f, indent=2)
                                print(f"OK statisticssaved: {stats_file}")
                                # ========================================================================
                                # 5. textdetailedstatistics
                                # ========================================================================
                                print("\n" + "="*80)
                                print("datastatisticstext")
                                print("="*80)
                                print(f"textsample count: {stats['total_samples']}")
                                print(f"total records: {stats['total_records']}")
                                print(f"uniquepeptidetext: {stats['unique_peptides']}")
                                print(f"uniqueMHCtext: {stats['unique_mhc']}")
                                print(f"uniquepeptide+MHCtext: {stats['unique_peptide_mhc']}")
                                print("\nstatistics by cancer type:")
                                for cancer, info in stats['cancer_types'].items():
                                    print(f" {cancer}:")
                                    print(f" - sample count: {info['samples']}")
                                    print(f" - recordtext: {info['records']}")
                                    print(f" - uniquepeptides: {info['unique_peptides']}")
                                    print(f" - uniqueMHC: {info['unique_mhc']}")
                                    print("\n" + "="*80)
                                    print("OK datatextcompleted!")
                                    print(f"completedtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                                    print("="*80)
                                    # ========================================================================
                                    # 6. generatetext
                                    # ========================================================================
                                    usage_guide = f"""
                                    # ============================================================================
                                    # text
                                    # ============================================================================

                                    ## filelist

                                    1. **unique_peptide_mhc_for_prediction.csv**
                                    - text: text
                                    - format: peptide_mhc_id, peptide, MHC
                                    - recordtext: {len(unique_peptide_mhc)}

                                    2. **peptide_full_mapping.csv**
                                    - text: textsample-microbe-peptide-MHCmappingtext
                                    - format: sample, cancer_type, peptide, MHC, microbes
                                    - recordtext: {len(full_data)}

                                    3. **peptide_full_mapping.pkl**
                                    - text: Pythontextloadtext(textCSVtext)

                                    4. **reverse_mapping.pkl**
                                    - text: textmappingtext(peptide+MHC -> sample columntext)

                                    5. **peptide_mapping_summary.json**
                                    - text: datastatistics

                                    ## text

                                    ### text1: text
                                    ```bash
                                    # text unique_peptide_mhc_for_prediction.csv textrowstext
                                    # textcompletedafter, resultstext peptide_mhc_id or peptide+MHC information
                                    ```

                                    ### text2: textresults(Python)
                                    ```python
                                    import pandas as pd
                                    import pickle

                                    # 1. loadtextresults(text peptide, MHC, immunogenicity_score column)
                                    prediction_results = pd.read_csv("immunogenicity_predictions.csv")

                                    # 2. loadcomplete mappingtext
                                    full_mapping = pd.read_csv("{mapping_file}")

                                    # 3. textdata
                                    # text1: textresultswithpeptide_mhc_id
                                    if 'peptide_mhc_id' in prediction_results.columns:
                                    unique_data = pd.read_csv("{unique_file}")
                                    # texttopeptideandMHC
                                    prediction_with_seq = prediction_results.merge(
                                    unique_data[['peptide_mhc_id', 'peptide', 'MHC']],                                     on='peptide_mhc_id'
                                    )
                                    # texttocomplete mapping
                                    final_results = full_mapping.merge(
                                    prediction_with_seq[['peptide', 'MHC', 'immunogenicity_score']],
                                    on=['peptide', 'MHC'],
                                    how='left'
                                    )

                                    # text2: textresultstextwithpeptideandMHC
                                    else:
                                    final_results = full_mapping.merge(
                                    prediction_results[['peptide', 'MHC', 'immunogenicity_score']],
                                    on=['peptide', 'MHC'],
                                    how='left'
                                    )

                                    # 4. saveresults
                                    final_results.to_csv("peptide_with_immunogenicity.csv", index=False)

                                    # 5. textsampleorcancer typetextstatistics
                                    sample_summary = final_results.groupby(['cancer_type', 'sample']).agg({{
                                    'immunogenicity_score': ['mean', 'median', 'max'],
                                    'peptide': 'count'
                                    }}).reset_index()
                                    ```

                                    ### text3: textmapping(text)
                                    ```python
                                    import pickle

                                    # loadtextmapping
                                    with open("{reverse_mapping_file}", 'rb') as f:
                                    reverse_mapping = pickle.load(f)

                                    # loadtextresults
                                    predictions = pd.read_csv("immunogenicity_predictions.csv")

                                    # foreachtextresultstextsampleinformation
                                    results_with_samples = []
                                    for _, row in predictions.iterrows():
                                    key = f"{{row['peptide']}}||{{row['MHC']}}"
                                    if key in reverse_mapping:
                                    for sample_info in reverse_mapping[key]:
                                    results_with_samples.append({{
                                    'peptide': row['peptide'],
                                    'MHC': row['MHC'],
                                    'immunogenicity_score': row['immunogenicity_score'],
                                    'sample': sample_info['sample'],
                                    'cancer_type': sample_info['cancer_type'],
                                    'microbes': sample_info['microbes']
                                    }})

                                    final_df = pd.DataFrame(results_with_samples)
                                    ```

                                    ## text

                                    1. **deduplicatetext**: textpeptide+MHCtextonlytext, textcalculatetext
                                    2. **mappingtext**: textpeptideandMHCtextcolumntextfortextrowsmapping
                                    3. **datatext**: textresultstext, textalltextsample-microbeinformation
                                    """
                                    usage_file = os.path.join(OUTPUT_DIR, "README.txt")
                                    with open(usage_file, 'w') as f:
                                        f.write(usage_guide)
                                        print(f"\nOK textsaved: {usage_file}")


# ============================================================================
# text
# ============================================================================

if __name__ == "__main__":
    process_all_data()

# textresultstextdata

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from datetime import datetime

# ========== text ==========
MAPPING_DIR = "/path/to/project/results_V3/summary/MTR_peptide_imm"
OUTPUT_DIR = "/path/to/project/results_V3/summary/MTR_peptide_imm"


def load_prediction_results(prediction_file):
    """loadtextresults(tsv), text HLA text MHC"""
    print(f"loadtextresults: {prediction_file}")

    required_cols = ['peptide', 'MHC'] # OK textrows

    try:
        df = pd.read_csv(prediction_file, sep='\t')
        print(f"OK load {len(df)} textresults")

        # OK processcolumn names: HLA -> MHC
        if 'HLA' in df.columns and 'MHC' not in df.columns:
            df = df.rename(columns={'HLA': 'MHC'})

            # OK checktextcolumn
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                raise ValueError(f"textresultstextcolumn: {missing}")

            print(f"OK textresultscolumn: {df.columns.tolist()}")
            return df

    except Exception as e:
        print(f"Failed error: {e}")
        return None


def merge_with_full_mapping(prediction_df, mapping_file):
    """text1: textcomplete mappingtext"""
    print("\ntext1: textcomplete mappingtext...")

    mapping_df = pd.read_pickle(mapping_file) if mapping_file.endswith(".pkl") else pd.read_csv(mapping_file)
    print(f"OK loadmappingtext: {len(mapping_df)} records")

    pred_cols = [c for c in prediction_df.columns if c not in ['peptide', 'MHC']]
    print(f"OK textcolumn: {pred_cols}")

    merge_cols = ['peptide', 'MHC'] + pred_cols
    result = mapping_df.merge(prediction_df[merge_cols], on=['peptide', 'MHC'], how='left')
    print(f"OK textafterrecordtext: {len(result)}")

    if pred_cols:
        unmatched = result[pred_cols[0]].isna().sum()
        if unmatched > 0:
            print(f"Warning️ warning: {unmatched} recordsnot foundtextresults(text {pred_cols[0]} text)")

            return result


def merge_with_reverse_mapping(prediction_df, reverse_mapping_file):
    """text2: text(text)"""
    print("\ntext2: textmappingtext...")

    with open(reverse_mapping_file, 'rb') as f:
        reverse_mapping = pickle.load(f)
        print(f"OK loadtextmapping: {len(reverse_mapping)} uniquepeptide+MHCtext")

        pred_cols = [c for c in prediction_df.columns if c not in ['peptide', 'MHC', 'peptide_mhc_id']]

        results_list = []
        for _, row in prediction_df.iterrows():
            key = f"{row['peptide']}||{row['MHC']}"
            if key in reverse_mapping:
                for sample_info in reverse_mapping[key]:
                    record = {
                        'sample': sample_info['sample'],
                        'cancer_type': sample_info['cancer_type'],
                        'peptide': row['peptide'],
                        'MHC': row['MHC'],
                        'microbes': sample_info['microbes']
                    }
                    for col in pred_cols:
                        record[col] = row[col]
                        results_list.append(record)

                        result = pd.DataFrame(results_list)
                        print(f"OK textafterrecordtext: {len(result)}")
                        return result


def generate_summary_statistics(merged_df):
    """generatetextstatistics(textfoundtext score column)"""
    print("\ngeneratetextstatistics...")

    score_cols = [c for c in merged_df.columns if ('score' in c.lower() or 'immuno' in c.lower())]
    if not score_cols:
        print("Warning️ not foundtextcolumn, skipstatisticsanalyze")
        return None

    score_col = score_cols[0]
    print(f"OK textcolumn: {score_col}")

    stats = {}

    sample_stats = merged_df.groupby(['cancer_type', 'sample']).agg({
        score_col: ['count', 'mean', 'median', 'std', 'min', 'max'],
        'peptide': 'nunique',
        'microbes': 'nunique'
    }).reset_index()

    sample_stats.columns = [
        'cancer_type', 'sample',
        'n_peptides', 'mean_score', 'median_score', 'std_score',
        'min_score', 'max_score',
        'n_unique_peptides', 'n_unique_microbes'
    ]
    stats['by_sample'] = sample_stats
    print(f"OK samplestatistics: {len(sample_stats)} samples")

    cancer_stats = merged_df.groupby('cancer_type').agg({
        score_col: ['count', 'mean', 'median', 'std'],
        'sample': 'nunique',
        'peptide': 'nunique',
        'microbes': 'nunique'
    }).reset_index()

    cancer_stats.columns = [
        'cancer_type',
        'n_records', 'mean_score', 'median_score', 'std_score',
        'n_samples', 'n_unique_peptides', 'n_unique_microbes'
    ]
    stats['by_cancer'] = cancer_stats
    print(f"OK cancer typestatistics: {len(cancer_stats)} cancer type")

    microbe_stats = merged_df.groupby('microbes').agg({
        score_col: ['count', 'mean', 'median'],
        'sample': 'nunique',
        'cancer_type': 'nunique'
    }).reset_index()

    microbe_stats.columns = [
        'microbes',
        'n_peptides', 'mean_score', 'median_score',
        'n_samples', 'n_cancer_types'
    ]
    stats['by_microbes_top20'] = microbe_stats.sort_values('n_peptides', ascending=False).head(20)
    print("OK microbestatistics: Top 20")

    return stats


def save_results(merged_df, stats, output_prefix):
    """saveresults"""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    main_file = os.path.join(OUTPUT_DIR, f"{output_prefix}_with_immunogenicity.csv")
    merged_df.to_csv(main_file, index=False)
    print(f"\nOK textfile: {main_file}")
    print(f" recordtext: {len(merged_df)}")

    pkl_file = os.path.join(OUTPUT_DIR, f"{output_prefix}_with_immunogenicity.pkl")
    merged_df.to_pickle(pkl_file)
    print(f"OK Picklefile: {pkl_file}")

    if stats:
        for name, df in stats.items():
            out = os.path.join(OUTPUT_DIR, f"{output_prefix}_summary_{name}.csv")
            df.to_csv(out, index=False)
            print(f"OK {name}: {out}")

            return main_file


def main(prediction_file, method='full', output_prefix='peptide_results'):
    print("="*80)
    print("textresults(Notebooktext)")
    print("="*80)
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"textfile: {prediction_file}")
    print(f"text: {method}")
    print(f"outputfirsttext: {output_prefix}")
    print("="*80)

    prediction_df = load_prediction_results(prediction_file)
    if prediction_df is None:
        return None

    if method == 'full':
        mapping_file = os.path.join(MAPPING_DIR, "peptide_full_mapping.pkl")
        if not os.path.exists(mapping_file):
            mapping_file = os.path.join(MAPPING_DIR, "peptide_full_mapping.csv")
            merged_df = merge_with_full_mapping(prediction_df, mapping_file)

        elif method == 'reverse':
            reverse_file = os.path.join(MAPPING_DIR, "reverse_mapping.pkl")
            merged_df = merge_with_reverse_mapping(prediction_df, reverse_file)

        else:
            raise ValueError("method onlytext 'full' or 'reverse'")

        stats = generate_summary_statistics(merged_df)
        output_file = save_results(merged_df, stats, output_prefix)

        print("\n" + "="*80)
        print("resultstext")
        print("="*80)
        print(f"inputrecordtext(text): {len(prediction_df)}")
        print(f"outputrecordtext(text): {len(merged_df)}")
        print(f"sample count: {merged_df['sample'].nunique()}")
        print(f"cancer typetext: {merged_df['cancer_type'].nunique()}")
        print(f"uniquepeptides: {merged_df['peptide'].nunique()}")
        print(f"uniquemicrobe: {merged_df['microbes'].nunique()}")
        print(f"textOutput files: {output_file}")
        print("="*80)

        return merged_df, stats


In [ ]:
prediction_file = "/path/to/project/results_V3/summary/MTR_peptide_imm/251213_peptides_with_immuno_tools.tsv"

merged_df, stats = main(
    prediction_file=prediction_file,
    method="reverse", # datatext "reverse"
    output_prefix="Imm_peptide" # textoutputfirsttext
)

merged_df.head()
